# **Data Raw Setup Notebook**

## Purpose

This notebook constructs the initial merged country-year dataset for the project.

The goal at this stage is to prepare and merge the primary quantitative source datasets into a common panel structure. This notebook does not aim to produce the final analysis-ready dataset. Final cleaning, imputation, enrichment with supplementary classifications, and feature engineering are handled in later stages.

The output of this notebook is an intermediate merged dataset saved in `data/processed_raw/`.

## Project Context

This project explores sustainable wellbeing across countries by combining indicators related to subjective wellbeing, inequality, economic development, resource use, energy consumption, and emissions.

The broader analytical goal is to investigate whether countries can achieve relatively high levels of wellbeing with lower environmental and material impact.

To support this, the first requirement is to construct a coherent country-year panel from several heterogeneous public datasets.

## Input Data Sources

This notebook uses the original primary datasets stored in `data/raw/`.

The main data sources include:

- World Happiness Index data
- GINI inequality data
- Our World in Data CO₂ and emissions data
- Our World in Data energy data
- Material footprint data

## Main Tasks

This notebook performs the following setup steps:

1. Load the original source datasets.
2. Standardise column names.
3. Inspect dataset shapes, year coverage, and key identifiers.
4. Reshape wide-format datasets into long country-year format where necessary.
5. Harmonise country names and country codes across sources.
6. Identify the common temporal window across datasets.
7. Filter datasets to the selected analysis period.
8. Merge the prepared datasets into a single country-year dataset.
9. Export the merged raw dataset to `data/processed_raw/`.

## Checks to Add or Maintain

The notebook should include explicit checks for:

- Whether each source dataset loads correctly.
- Whether country and year identifiers are available after preparation.
- Whether wide-to-long transformations produce the expected structure.
- Whether merge keys are unique where they need to be unique.
- Whether merges introduce unexpected row multiplication.
- Whether the selected year range is consistently applied.
- Whether the final merged dataset has unique country-year observations.
- Whether the exported file can be reloaded successfully.

## Output

The output of this notebook is:

```text
data/processed_raw/global_sustainability_wellbeing_resource_data_raw.csv
```

This file is an intermediate merged dataset. It is not yet the final analysis-ready dataset.

We gather panel data (country/year) on wellbeing, inequality, material consumption and emissions from various official sources listed below:

## Data Sources

| Dataset | Coverage | Source | Purpose in Project | Original Format |
|---|---|---|---|---|
| OWID CO₂ Data | Annual country-level data (varies by variable; broad coverage 1900–2023) | https://github.com/owid/co2-data | Base dataset | Long |
| OWID Energy Data | Annual country-level energy indicators | https://github.com/owid/energy-data | Supplementary energy transition indicators | Long |
| Material Footprint per Capita | 2012–2021 | https://www.kaggle.com/datasets/iamsouravbanerjee/material-footprint-per-capita-by-country | Per-capita material consumption indicator | Wide |
| World Bank GINI Index | 1992–2025 (sparse by country) | https://data.worldbank.org/indicator/SI.POV.GINI | Income inequality indicator | Wide |
| World Happiness Index | 2013–2023 | https://www.kaggle.com/datasets/simonaasm/world-happiness-index-by-reports-2013-2023 | Subjective wellbeing indicator | Long |

## Temporal Overlap

The main datasets overlap consistently between **2013 and 2021**, giving a usable multi-year comparison window for merged analysis.

---

## Core Theoretical Variables from OWID CO₂ and Energy Datasets

| Variable | Description | Analytical Purpose |
|---|---|---|
| `population` | Total population of the country | Useful for weighting and validating per-capita indicators |
| `gdp` | Gross Domestic Product | Main macroeconomic control variable |
| `consumption_co2_per_capita` | Consumption-based CO₂ emissions per capita | Key variable linking emissions to lifestyles and consumption patterns |
| `co2_per_capita` | Production-based CO₂ emissions per capita | Used for comparison with consumption emissions and emissions-gap calculations |
| `renewables_share_energy` | Share of primary energy consumption from renewable sources | Indicator of energy transition and decarbonization |
| `energy_per_capita` | Primary energy consumption per capita | Proxy for energy intensity and resource use |

---

## Identifier and Merge Variables

| Variable | Description | Role in Merge |
|---|---|---|
| `iso_code` | ISO 3-letter country code | Main country-level merge key |
| `country` | Full country name | Human-readable country identifier |
| `year` | Observation year | Temporal merge key for panel structure |

---

## Conceptual Focus of the Project

This project investigates whether countries can achieve relatively high levels of wellbeing while maintaining lower levels of environmental and material throughput.

The analysis combines:
- wellbeing indicators,
- inequality measures,
- emissions data,
- renewable energy transition metrics,
- and material footprint indicators

to explore possible forms of:
- sustainable wellbeing,
- ecological efficiency,
- and partial decoupling between quality of life and material consumption.

### Strategy summary:
1. Import the raw datasets independently.
2. Convert wide data to long format.
3. Check temporal coverage per dataset.
4. Keep only overlapping window accross all datasets
5. Check and normalise merge keys accross datasets (iso, country, year).
6. From OWID CO2 and Energy data, pick selected columns to keep.
7. Merge by country (iso) and year keys to create the full raw dataset to be cleaned.

## 1. Import the necessary libraries

In [1]:
import pandas as pd
import re
from difflib import get_close_matches
import numpy as np

In [2]:
# Make imports from the scr/ directory work.
import sys
from pathlib import Path

# Add project root to path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

## 2. Load the data

In [3]:
from src.io import load_csv
from src.config import CO2_PATH, ENERGY_PATH, HAPPINESS_PATH, MATERIAL_FOOTPRINT_PATH, GINI_PATH

# Long format data
owid_co2_df = load_csv(CO2_PATH)
owid_energy_df = load_csv(ENERGY_PATH)
hi_df = load_csv(HAPPINESS_PATH)

# Wide format data has to be converted to long prior to any merging
mf_df_wide = load_csv(MATERIAL_FOOTPRINT_PATH)
gini_df_wide = load_csv(GINI_PATH)

We double check long/wide data formats have been correctly identified, and look at dataset shapes.

In [4]:
print("OWID CO2 DataFrame shape:", owid_co2_df.shape)
display(owid_co2_df.head(3))

print("OWID Energy DataFrame shape:", owid_energy_df.shape)
display(owid_energy_df.head(3))

print("World Happiness Index DataFrame shape:", hi_df.shape)
display(hi_df.head(3))

print("Material Footprint DataFrame shape:", mf_df_wide.shape)
display(mf_df_wide.head(3))

print("Gini DataFrame shape:", gini_df_wide.shape)
display(gini_df_wide.head(3))

OWID CO2 DataFrame shape: (50411, 79)


,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


OWID Energy DataFrame shape: (23377, 130)


,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN


World Happiness Index DataFrame shape: (1670, 4)


,Country,Year,Index,Rank
0,Afghanistan,2013,4.040,143.0
1,Afghanistan,2015,3.575,153.0
2,Afghanistan,2016,3.360,154.0


Material Footprint DataFrame shape: (195, 39)


,ISO3,Country,Continent,Hemisphere,Human Development Groups,UNDP Developing Regions,HDI Rank (2021),Material footprint per capita (tonnes) (1990),Material footprint per capita (tonnes) (1991),Material footprint per capita (tonnes) (1992),...,Material footprint per capita (tonnes) (2012),Material footprint per capita (tonnes) (2013),Material footprint per capita (tonnes) (2014),Material footprint per capita (tonnes) (2015),Material footprint per capita (tonnes) (2016),Material footprint per capita (tonnes) (2017),Material footprint per capita (tonnes) (2018),Material footprint per capita (tonnes) (2019),Material footprint per capita (tonnes) (2020),Material footprint per capita (tonnes) (2021)
0,AFG,Afghanistan,Asia,Northern Hemisphere,Low,SA,180.0,2.33,2.28,2.35,...,1.86,1.88,1.66,1.62,1.66,1.41,1.32,1.38,1.38,1.38
1,AGO,Angola,Africa,Southern Hemisphere,Medium,SSA,148.0,2.44,2.66,4.67,...,4.09,4.53,3.97,3.59,2.79,2.64,2.28,2.18,2.18,2.18
2,ALB,Albania,Europe,Northern Hemisphere,High,ECA,67.0,6.63,5.91,5.65,...,12.44,11.49,13.14,12.61,14.39,14.46,12.85,12.96,12.96,12.96


Gini DataFrame shape: (266, 71)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
print(mf_df_wide.columns)
print(gini_df_wide.columns)

Index(['ISO3', 'Country', 'Continent', 'Hemisphere',
       'Human Development Groups', 'UNDP Developing Regions',
       'HDI Rank (2021)', 'Material footprint per capita (tonnes) (1990)',
       'Material footprint per capita (tonnes) (1991)',
       'Material footprint per capita (tonnes) (1992)',
       'Material footprint per capita (tonnes) (1993)',
       'Material footprint per capita (tonnes) (1994)',
       'Material footprint per capita (tonnes) (1995)',
       'Material footprint per capita (tonnes) (1996)',
       'Material footprint per capita (tonnes) (1997)',
       'Material footprint per capita (tonnes) (1998)',
       'Material footprint per capita (tonnes) (1999)',
       'Material footprint per capita (tonnes) (2000)',
       'Material footprint per capita (tonnes) (2001)',
       'Material footprint per capita (tonnes) (2002)',
       'Material footprint per capita (tonnes) (2003)',
       'Material footprint per capita (tonnes) (2004)',
       'Material footprint

## 3. Standardise column names in the wide dataframes

It makes sense before going wide-to-long, to standardize column names accross datasets. Doing this first saves time not only with wide datasets, but also ther unconventional column names in the World Happiness Index dataset.

A `clean_colum_names` function is created to accomodate this:

In [6]:
from src.utils import clean_column_names

In [7]:
# Apply clean_column_names function and print the cleaned dataframes

owid_co2_df = clean_column_names(owid_co2_df)
owid_energy_df = clean_column_names(owid_energy_df)
hi_df = clean_column_names(hi_df)
mf_df_wide = clean_column_names(mf_df_wide)
gini_df_wide = clean_column_names(gini_df_wide)

display(owid_co2_df.head(3))
display(owid_energy_df.head(3))
display(hi_df.head(3))
display(mf_df_wide.head(3))
display(gini_df_wide.head(3))

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN


,country,year,index,rank
0,Afghanistan,2013,4.040,143.0
1,Afghanistan,2015,3.575,153.0
2,Afghanistan,2016,3.360,154.0


,iso3,country,continent,hemisphere,human_development_groups,undp_developing_regions,hdi_rank_2021,material_footprint_per_capita_tonnes_1990,material_footprint_per_capita_tonnes_1991,material_footprint_per_capita_tonnes_1992,...,material_footprint_per_capita_tonnes_2012,material_footprint_per_capita_tonnes_2013,material_footprint_per_capita_tonnes_2014,material_footprint_per_capita_tonnes_2015,material_footprint_per_capita_tonnes_2016,material_footprint_per_capita_tonnes_2017,material_footprint_per_capita_tonnes_2018,material_footprint_per_capita_tonnes_2019,material_footprint_per_capita_tonnes_2020,material_footprint_per_capita_tonnes_2021
0,AFG,Afghanistan,Asia,Northern Hemisphere,Low,SA,180.0,2.33,2.28,2.35,...,1.86,1.88,1.66,1.62,1.66,1.41,1.32,1.38,1.38,1.38
1,AGO,Angola,Africa,Southern Hemisphere,Medium,SSA,148.0,2.44,2.66,4.67,...,4.09,4.53,3.97,3.59,2.79,2.64,2.28,2.18,2.18,2.18
2,ALB,Albania,Europe,Northern Hemisphere,High,ECA,67.0,6.63,5.91,5.65,...,12.44,11.49,13.14,12.61,14.39,14.46,12.85,12.96,12.96,12.96


,country_name,country_code,indicator_name,indicator_code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,unnamed_70
0,Aruba,ABW,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


We then manually change the remaining iso3 in mf_df_wide, and counry_name, country_code columns in gini_df_wide to match all other dataframes.

In [8]:
mf_df_wide = mf_df_wide.rename(columns={'iso3': 'iso_code'})
gini_df_wide = gini_df_wide.rename(columns={'country_name': 'country', 'country_code': 'iso_code'})

print(mf_df_wide.columns)
print(gini_df_wide.columns)

Index(['iso_code', 'country', 'continent', 'hemisphere',
       'human_development_groups', 'undp_developing_regions', 'hdi_rank_2021',
       'material_footprint_per_capita_tonnes_1990',
       'material_footprint_per_capita_tonnes_1991',
       'material_footprint_per_capita_tonnes_1992',
       'material_footprint_per_capita_tonnes_1993',
       'material_footprint_per_capita_tonnes_1994',
       'material_footprint_per_capita_tonnes_1995',
       'material_footprint_per_capita_tonnes_1996',
       'material_footprint_per_capita_tonnes_1997',
       'material_footprint_per_capita_tonnes_1998',
       'material_footprint_per_capita_tonnes_1999',
       'material_footprint_per_capita_tonnes_2000',
       'material_footprint_per_capita_tonnes_2001',
       'material_footprint_per_capita_tonnes_2002',
       'material_footprint_per_capita_tonnes_2003',
       'material_footprint_per_capita_tonnes_2004',
       'material_footprint_per_capita_tonnes_2005',
       'material_footprint_per_c

In [9]:
# Quick check - What is the 'unnamed_70' column in the gini_df_wide dataframe? It seems to be empty
# A: It's just an empty column - can drop.

print(gini_df_wide.shape)
print(f" Missing values in the unnamed_70 column: {gini_df_wide['unnamed_70'].isna().sum()}")

(266, 71)
 Missing values in the unnamed_70 column: 266


## 4. Transform wide data to long format

We now write a wide_to_long function that can be applied to both mf and gini DataFrames and gives us a long format with standardized column names for country, year and value.

In [10]:
from src.utils import wide_to_long

We now apply the function to both datasets, and check the resulting long format DataFrames.

In [11]:
mf_id_vars = [
    "iso_code",
    "country",
    "continent",
    "hemisphere",
    "human_development_groups",
    "undp_developing_regions",
    "hdi_rank_2021",
]

mf_df = wide_to_long(
    df=mf_df_wide,
    id_vars=mf_id_vars,
    year_pattern=r"material_footprint_per_capita_tonnes_(\d{4})",
    value_name="material_footprint_per_capita",
)

In [12]:
gini_id_vars = ["country", "iso_code", "indicator_name", "indicator_code"]

gini_df = wide_to_long(
    df=gini_df_wide,
    id_vars=gini_id_vars,
    year_pattern=r"^(\d{4})$",
    value_name="gini_index",
)


print(mf_df.shape)
print(gini_df.shape)
display(mf_df.head())
display(gini_df.head())

(6240, 9)
(17556, 6)


,iso_code,country,continent,hemisphere,human_development_groups,undp_developing_regions,hdi_rank_2021,year,material_footprint_per_capita
0,AFG,Afghanistan,Asia,Northern Hemisphere,Low,SA,180.0,1990,2.33
1,AGO,Angola,Africa,Southern Hemisphere,Medium,SSA,148.0,1990,2.44
2,ALB,Albania,Europe,Northern Hemisphere,High,ECA,67.0,1990,6.63
3,AND,Andorra,Europe,Northern Hemisphere,Very High,NaN,40.0,1990,NaN
4,ARE,United Arab Emirates,Asia,Northern Hemisphere,Very High,AS,26.0,1990,64.75


,country,iso_code,indicator_name,indicator_code,year,gini_index
0,Aruba,ABW,Gini index,SI.POV.GINI,1960,NaN
1,Africa Eastern and Southern,AFE,Gini index,SI.POV.GINI,1960,NaN
2,Afghanistan,AFG,Gini index,SI.POV.GINI,1960,NaN
3,Africa Western and Central,AFW,Gini index,SI.POV.GINI,1960,NaN
4,Angola,AGO,Gini index,SI.POV.GINI,1960,NaN


## 5. Standardize remaining column names and merge keys - Happiness Index

We must also deal with the Happiness Index dataset, where the column of interest has name "index", which is not comprehensive once we merge with other data. Also, this dataset does not contain the iso_code column we intend to use as the primary merge column later on.
1. We rename the index column to happiness_index.
2. We extract the iso_codes from the owid_co2_df and add them here to match other datasets.

In [13]:
# Step 1: Rename the 'index' column to 'happiness_index'
hi_df = hi_df.rename(columns={'index': 'happiness_index', 'rank': 'happiness_index_rank'})

Step 2 is more challenging and brings up a potential issue with the data: What is the country coverage of each dataset?

We write a function that takes two DataFrames, the names of their country columns, and a name for each and checks for missmatches in the countries that are covered by each.

In [14]:
from src.utils import compare_values

In [15]:
# Compare Happiness Index vs OWID
compare_values(
    owid_co2_df,
    hi_df,
    col1="country",
    col2="country",
    name1="OWID CO2",
    name2="Happiness"
)

Unique values | OWID CO2: 254 | Happiness: 167

Shared values: 159

[1/2] In OWID CO2 but missing from Happiness (95):
['Africa', 'Africa (GCP)', 'Andorra', 'Anguilla', 'Antarctica', 'Antigua and Barbuda', 'Aruba', 'Asia', 'Asia (GCP)', 'Asia (excl. China and India)', 'Bahamas', 'Barbados', 'Bermuda', 'Bonaire Sint Eustatius and Saba', 'British Virgin Islands', 'Brunei', 'Cape Verde', 'Central America (GCP)', 'Christmas Island', 'Cook Islands', "Cote d'Ivoire", 'Cuba', 'Curacao', 'Democratic Republic of Congo', 'Dominica', 'East Timor', 'Equatorial Guinea', 'Eritrea', 'Europe', 'Europe (GCP)', 'Europe (excl. EU-27)', 'Europe (excl. EU-28)', 'European Union (27)', 'European Union (28)', 'Faroe Islands', 'Fiji', 'French Polynesia', 'Greenland', 'Grenada', 'Guinea-Bissau', 'Guyana', 'High-income countries', 'International aviation', 'International shipping', 'Kiribati', 'Kuwaiti Oil Fires', 'Kuwaiti Oil Fires (GCP)', 'Least developed countries (Jones et al.)', 'Liechtenstein', 'Low-income

{'shared': ['Afghanistan',
  'Albania',
  'Algeria',
  'Angola',
  'Argentina',
  'Armenia',
  'Australia',
  'Austria',
  'Azerbaijan',
  'Bahrain',
  'Bangladesh',
  'Belarus',
  'Belgium',
  'Belize',
  'Benin',
  'Bhutan',
  'Bolivia',
  'Bosnia and Herzegovina',
  'Botswana',
  'Brazil',
  'Bulgaria',
  'Burkina Faso',
  'Burundi',
  'Cambodia',
  'Cameroon',
  'Canada',
  'Central African Republic',
  'Chad',
  'Chile',
  'China',
  'Colombia',
  'Comoros',
  'Congo',
  'Costa Rica',
  'Croatia',
  'Cyprus',
  'Czechia',
  'Denmark',
  'Djibouti',
  'Dominican Republic',
  'Ecuador',
  'Egypt',
  'El Salvador',
  'Estonia',
  'Eswatini',
  'Ethiopia',
  'Finland',
  'France',
  'Gabon',
  'Gambia',
  'Georgia',
  'Germany',
  'Ghana',
  'Greece',
  'Guatemala',
  'Guinea',
  'Haiti',
  'Honduras',
  'Hong Kong',
  'Hungary',
  'Iceland',
  'India',
  'Indonesia',
  'Iran',
  'Iraq',
  'Ireland',
  'Israel',
  'Italy',
  'Jamaica',
  'Japan',
  'Jordan',
  'Kazakhstan',
  'Kenya',

This highlights two important and relevant facts about the dataset structures that we still need to deal with:
1. The same country may have different names, like Turkiye and Turkey.

    - *Note: this is only problematic in the happiness index data which doesnt have an iso_code column to start with*
    
2. The OWID CO2 dataset contains non country columns like Africa, Asia, Least developed countries etc.

In [16]:
# Which 'countries' values are missing iso_code in the OWID dataset?
# This also gives us a way to identify non-country identifiers in the OWID dataset which might be useful later.

missing_iso = owid_co2_df[
    owid_co2_df["iso_code"].isna() |
    (owid_co2_df["iso_code"] == "")
]

print(missing_iso["country"].unique())

<StringArray>
[                                  'Africa',
                             'Africa (GCP)',
                                     'Asia',
                               'Asia (GCP)',
             'Asia (excl. China and India)',
                    'Central America (GCP)',
                                   'Europe',
                             'Europe (GCP)',
                     'Europe (excl. EU-27)',
                     'Europe (excl. EU-28)',
                      'European Union (27)',
                      'European Union (28)',
                    'High-income countries',
                   'International aviation',
                   'International shipping',
                                   'Kosovo',
                        'Kuwaiti Oil Fires',
                  'Kuwaiti Oil Fires (GCP)',
 'Least developed countries (Jones et al.)',
                     'Low-income countries',
            'Lower-middle-income countries',
                        'Middle East (GCP

In [17]:
# Step 2: Grab iso_code column from owid_co2_df

# 1. Create a clean 1-to-1 mapping from the OWID dataset
# Drop duplicates since OWID has multiple rows per country and dropna since there are non-country entries which will not have an iso_code as confirmed by the code above
iso_mapping = owid_co2_df[["country", "iso_code"]].drop_duplicates().dropna()

missmatch = compare_values(
    iso_mapping,
    hi_df,
    col1="country",
    col2="country",
    name1="ISO_MAPPING",
    name2="Happiness"
)

Unique values | ISO_MAPPING: 218 | Happiness: 167

Shared values: 158

[1/2] In ISO_MAPPING but missing from Happiness (60):
['Andorra', 'Anguilla', 'Antarctica', 'Antigua and Barbuda', 'Aruba', 'Bahamas', 'Barbados', 'Bermuda', 'Bonaire Sint Eustatius and Saba', 'British Virgin Islands', 'Brunei', 'Cape Verde', 'Christmas Island', 'Cook Islands', "Cote d'Ivoire", 'Cuba', 'Curacao', 'Democratic Republic of Congo', 'Dominica', 'East Timor', 'Equatorial Guinea', 'Eritrea', 'Faroe Islands', 'Fiji', 'French Polynesia', 'Greenland', 'Grenada', 'Guinea-Bissau', 'Guyana', 'Kiribati', 'Liechtenstein', 'Macao', 'Marshall Islands', 'Micronesia (country)', 'Monaco', 'Montserrat', 'Nauru', 'New Caledonia', 'Niue', 'North Korea', 'Palau', 'Papua New Guinea', 'Saint Helena', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Pierre and Miquelon', 'Saint Vincent and the Grenadines', 'Samoa', 'San Marino', 'Sao Tome and Principe', 'Seychelles', 'Sint Maarten (Dutch part)', 'Solomon Islands', 'Tonga', 'Tur

We write a function to check close matches between the identified countries that are only in either of the compared column-dataset pairs but not the other.

In [18]:
from src.utils import check_close_matches

In [19]:
check_close_matches(
    missmatch["only_in_df1"],
    missmatch["only_in_df2"]
)

North Korea -> ['North Cyprus']
Turkey -> ['Turkiye']


So the only country that is in both datasets but with different name is Turekey. We change this value manually to match the OWID dataset nomenclature (in English): 'Turkey' and check that Turkey now appears in the shared countries of the two datasets.

In [20]:
hi_df.loc[hi_df["country"] == "Turkiye", "country"] = "Turkey"

compare_values(
    iso_mapping,
    hi_df,
    col1="country",
    col2="country",
    name1="ISO_MAPPING",
    name2="Happiness"
)

Unique values | ISO_MAPPING: 218 | Happiness: 167

Shared values: 159

[1/2] In ISO_MAPPING but missing from Happiness (59):
['Andorra', 'Anguilla', 'Antarctica', 'Antigua and Barbuda', 'Aruba', 'Bahamas', 'Barbados', 'Bermuda', 'Bonaire Sint Eustatius and Saba', 'British Virgin Islands', 'Brunei', 'Cape Verde', 'Christmas Island', 'Cook Islands', "Cote d'Ivoire", 'Cuba', 'Curacao', 'Democratic Republic of Congo', 'Dominica', 'East Timor', 'Equatorial Guinea', 'Eritrea', 'Faroe Islands', 'Fiji', 'French Polynesia', 'Greenland', 'Grenada', 'Guinea-Bissau', 'Guyana', 'Kiribati', 'Liechtenstein', 'Macao', 'Marshall Islands', 'Micronesia (country)', 'Monaco', 'Montserrat', 'Nauru', 'New Caledonia', 'Niue', 'North Korea', 'Palau', 'Papua New Guinea', 'Saint Helena', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Pierre and Miquelon', 'Saint Vincent and the Grenadines', 'Samoa', 'San Marino', 'Sao Tome and Principe', 'Seychelles', 'Sint Maarten (Dutch part)', 'Solomon Islands', 'Tonga', 'Tur

{'shared': ['Afghanistan',
  'Albania',
  'Algeria',
  'Angola',
  'Argentina',
  'Armenia',
  'Australia',
  'Austria',
  'Azerbaijan',
  'Bahrain',
  'Bangladesh',
  'Belarus',
  'Belgium',
  'Belize',
  'Benin',
  'Bhutan',
  'Bolivia',
  'Bosnia and Herzegovina',
  'Botswana',
  'Brazil',
  'Bulgaria',
  'Burkina Faso',
  'Burundi',
  'Cambodia',
  'Cameroon',
  'Canada',
  'Central African Republic',
  'Chad',
  'Chile',
  'China',
  'Colombia',
  'Comoros',
  'Congo',
  'Costa Rica',
  'Croatia',
  'Cyprus',
  'Czechia',
  'Denmark',
  'Djibouti',
  'Dominican Republic',
  'Ecuador',
  'Egypt',
  'El Salvador',
  'Estonia',
  'Eswatini',
  'Ethiopia',
  'Finland',
  'France',
  'Gabon',
  'Gambia',
  'Georgia',
  'Germany',
  'Ghana',
  'Greece',
  'Guatemala',
  'Guinea',
  'Haiti',
  'Honduras',
  'Hong Kong',
  'Hungary',
  'Iceland',
  'India',
  'Indonesia',
  'Iran',
  'Iraq',
  'Ireland',
  'Israel',
  'Italy',
  'Jamaica',
  'Japan',
  'Jordan',
  'Kazakhstan',
  'Kenya',

Now that country names are correctly matched, we give countries in the happiness index dataset their corresponding ISO codes.

In [21]:
# 2. Left merge the ISO codes into your Happiness Index dataframe
hi_df_iso = hi_df.merge(iso_mapping, on="country", how="left")

print(f"Out of {hi_df_iso[['country']].drop_duplicates().shape[0]} countries, {hi_df_iso[['iso_code']].drop_duplicates().dropna().shape[0]} have ISO codes.")

# Which countries dont have iso codes?
print(f"The countries that dont have iso_codes are: {hi_df_iso[hi_df_iso['iso_code'].isna()]['country'].unique()}")
display(hi_df_iso[['country', 'iso_code']].drop_duplicates())

Out of 167 countries, 159 have ISO codes.
The countries that dont have iso_codes are: <StringArray>
['Congo Brazzaville',    'Congo Kinshasa',       'Ivory Coast',
            'Kosovo',      'North Cyprus',       'Puerto Rico',
        'Somaliland',         'Swaziland']
Length: 8, dtype: str


,country,iso_code
0,Afghanistan,AFG
10,Albania,ALB
20,Algeria,DZA
30,Angola,AGO
40,Argentina,ARG
...,...,...
1620,Venezuela,VEN
1630,Vietnam,VNM
1640,Yemen,YEM
1650,Zambia,ZMB


# 6. Drop non-country entries in the 'country' column in the basis dataset

We had previously identified unwanted groupings by continent and other criteria under the country column in the OWID CO2 dataset. These are not necessary to keep at this point and can be computed based on country-level metrics as aggregate metrics. Therefore we identify these and drop them.

In [22]:
# Recall:
# Which 'country' values are missing iso_code in the OWID dataset?

missing_iso = owid_co2_df[
    owid_co2_df["iso_code"].isna() |
    (owid_co2_df["iso_code"] == "")
]

print(missing_iso["country"].unique())

# We now wish to drop rows that do not have an iso_code
owid_co2_df_clean = owid_co2_df.dropna(subset=["iso_code"])

missing_iso_clean = owid_co2_df_clean[
    owid_co2_df_clean["iso_code"].isna() |
    (owid_co2_df_clean["iso_code"] == "")
]

print(missing_iso_clean["country"].unique())

<StringArray>
[                                  'Africa',
                             'Africa (GCP)',
                                     'Asia',
                               'Asia (GCP)',
             'Asia (excl. China and India)',
                    'Central America (GCP)',
                                   'Europe',
                             'Europe (GCP)',
                     'Europe (excl. EU-27)',
                     'Europe (excl. EU-28)',
                      'European Union (27)',
                      'European Union (28)',
                    'High-income countries',
                   'International aviation',
                   'International shipping',
                                   'Kosovo',
                        'Kuwaiti Oil Fires',
                  'Kuwaiti Oil Fires (GCP)',
 'Least developed countries (Jones et al.)',
                     'Low-income countries',
            'Lower-middle-income countries',
                        'Middle East (GCP

We have correctly removed the rows with missing iso_code. Meaning we now only have rows corresponding to iso-coded countries - as desired.

## 7. Identify year overlap window over all datasets

1. Identify each dataset's year range and coverage.
2. Keep only the common year range across all datasets for the merged dataset.

In [23]:
# We look at each dataframe's year coverage
print(f"\nOWID CO2 DataFrame covers from year {owid_co2_df_clean['year'].min()} to year {owid_co2_df_clean['year'].max()}")
print(f"\nOWID Energy DataFrame covers from year {owid_energy_df['year'].min()} to year {owid_energy_df['year'].max()}")
print(f"\nHappiness Index DataFrame covers from year {hi_df_iso['year'].min()} to year {hi_df_iso['year'].max()}")
print(f"\nMaterial Footprint DataFrame covers from year {mf_df['year'].min()} to year {mf_df['year'].max()}")
print(f"\nGini DataFrame covers from year {gini_df['year'].min()} to year {gini_df['year'].max()}")

# We confirm the common year window
print(f"\nThe common year window to all datasets is from {np.max(np.array([owid_co2_df_clean['year'].min(), owid_energy_df['year'].min(), hi_df_iso['year'].min(), mf_df['year'].min(), gini_df['year'].min()]))} to {np.min(np.array([owid_co2_df_clean['year'].max(), owid_energy_df['year'].max(), hi_df_iso['year'].max(), mf_df['year'].max(), gini_df['year'].max()]))}")


OWID CO2 DataFrame covers from year 1750 to year 2024

OWID Energy DataFrame covers from year 1900 to year 2025

Happiness Index DataFrame covers from year 2013 to year 2023

Material Footprint DataFrame covers from year 1990 to year 2021

Gini DataFrame covers from year 1960 to year 2025

The common year window to all datasets is from 2013 to 2021


In [24]:
# We double check the year column's types
print(owid_co2_df_clean['year'].dtype)
print(owid_energy_df['year'].dtype)
print(hi_df_iso['year'].dtype)
print(mf_df['year'].dtype)
print(gini_df['year'].dtype)

int64
int64
int64
int64
int64


Function to reduce year coverage to a given window:

In [25]:
from src.utils import filter_year_range

START_YEAR = 2013
END_YEAR = 2021

In [26]:
# Apply to all datasets
owid_co2_df_clean = filter_year_range(owid_co2_df_clean)
owid_energy_df_clean = filter_year_range(owid_energy_df)
hi_df_clean = filter_year_range(hi_df_iso)
mf_df_clean = filter_year_range(mf_df)
gini_df_clean = filter_year_range(gini_df)


# Quick verification
print(
    f"Filtered common window: "
    f"{START_YEAR}-{END_YEAR}"
)

print(
    f"\nOWID CO2: "
    f"{owid_co2_df_clean['year'].min()} - "
    f"{owid_co2_df_clean['year'].max()}"
)

print(
    f"OWID Energy: "
    f"{owid_energy_df_clean['year'].min()} - "
    f"{owid_energy_df_clean['year'].max()}"
)

print(
    f"Happiness Index: "
    f"{hi_df_clean['year'].min()} - "
    f"{hi_df_clean['year'].max()}"
)

print(
    f"Material Footprint: "
    f"{mf_df_clean['year'].min()} - "
    f"{mf_df_clean['year'].max()}"
)

print(
    f"Gini: "
    f"{gini_df_clean['year'].min()} - "
    f"{gini_df_clean['year'].max()}"
)

print(
    f"\nOWID CO2 has {len(owid_co2_df_clean)} rows"
    f"\nOWID Energy has {len(owid_energy_df_clean)} rows"
    f"\nMaterial Footprint has {len(mf_df_clean)} rows"
    f"\nGini has {len(gini_df_clean)} rows"
    f"\nHappiness Index has {len(hi_df_clean)} rows"
)   

Filtered common window: 2013-2021

OWID CO2: 2013 - 2021
OWID Energy: 2013 - 2021
Happiness Index: 2013 - 2021
Material Footprint: 2013 - 2021
Gini: 2013 - 2021

OWID CO2 has 1962 rows
OWID Energy has 2700 rows
Material Footprint has 1755 rows
Gini has 2394 rows
Happiness Index has 1336 rows


### 7.1 Check for duplicate keys

In [27]:
from src.utils import check_duplicate_keys

check_duplicate_keys(hi_df_clean, ["iso_code", "year"], "hi_df_clean")
check_duplicate_keys(gini_df_clean, ["iso_code", "year"], "gini_df_clean")
check_duplicate_keys(owid_co2_df_clean, ["iso_code", "year"], "owid_co2_df_clean")
check_duplicate_keys(owid_energy_df_clean, ["iso_code", "year"], "owid_energy_df_clean")
check_duplicate_keys(mf_df_clean, ["iso_code", "year"], "mf_df_clean")

64 rows with duplicate keys in hi_df_clean for ['iso_code', 'year']
                country  year  happiness_index  happiness_index_rank iso_code
330   Congo Brazzaville  2013            4.297                 129.0      NaN
340      Congo Kinshasa  2013            4.578                 117.0      NaN
700         Ivory Coast  2013              NaN                   NaN      NaN
760              Kosovo  2013            5.222                  83.0      NaN
1090       North Cyprus  2013            5.463                  69.0      NaN
...                 ...   ...              ...                   ...      ...
767              Kosovo  2021            6.372                  33.0      NaN
1097       North Cyprus  2021            5.536                  74.0      NaN
1217        Puerto Rico  2021              NaN                   NaN      NaN
1347         Somaliland  2021              NaN                   NaN      NaN
1427          Swaziland  2021            4.308                 130.0      

,iso_code,country,continent,hemisphere,human_development_groups,undp_developing_regions,hdi_rank_2021,year,material_footprint_per_capita


The duplicate counts are consistent with all missing iso_code rows collapsing into the same key per year. We isolate iso_code.isna(), list their country names, then test whether those countries would ever match the CO₂ left-merge base.

In [28]:
# Inspect rows with missing ISO codes in each prepared dataset

# datasets = {
#     "hi_df_clean": hi_df_clean,
#     "gini_df_clean": gini_df_clean,
#     "owid_co2_df_clean": owid_co2_df_clean,
#     "owid_energy_df_clean": owid_energy_df_clean,
#     "mf_df_clean": mf_df_clean,
# }

datasets_missing_iso = {
    "hi_df_clean": hi_df_clean,
    "owid_energy_df_clean": owid_energy_df_clean
}

for name, df in datasets_missing_iso.items():
    missing_iso = df[df["iso_code"].isna()].copy()

    print(f"\n{name}")
    print(f"Rows with missing iso_code: {len(missing_iso)}")
    print(f"Unique countries/entities with missing iso_code: {missing_iso['country'].nunique()}")

    display(
        missing_iso[["country", "year"]]
        .drop_duplicates()
        .sort_values(["country", "year"])
        .head(50)
    )


hi_df_clean
Rows with missing iso_code: 64
Unique countries/entities with missing iso_code: 8


,country,year
330,Congo Brazzaville,2013
331,Congo Brazzaville,2015
332,Congo Brazzaville,2016
333,Congo Brazzaville,2017
334,Congo Brazzaville,2018
335,Congo Brazzaville,2019
336,Congo Brazzaville,2020
337,Congo Brazzaville,2021
340,Congo Kinshasa,2013
341,Congo Kinshasa,2015



owid_energy_df_clean
Rows with missing iso_code: 720
Unique countries/entities with missing iso_code: 88


,country,year
13,ASEAN (Ember),2013
14,ASEAN (Ember),2014
15,ASEAN (Ember),2015
16,ASEAN (Ember),2016
17,ASEAN (Ember),2017
18,ASEAN (Ember),2018
19,ASEAN (Ember),2019
20,ASEAN (Ember),2020
21,ASEAN (Ember),2021
264,Africa,2013


Missing iso_code from Energy dataset are country groupings which are correctly missing an ISO. Nothing to do here.

In the Happiness index case we have legitimate missing countries. Some are common and are filled in manually, other are less standard so we want to check wheter they exist in the original dataset.

In [29]:
hi_iso_fixes = {
    "Congo Brazzaville": "COG",  # Republic of the Congo
    "Congo Kinshasa": "COD",     # Democratic Republic of the Congo
    "Ivory Coast": "CIV",        # Côte d'Ivoire
    "Puerto Rico": "PRI",
}

hi_nonstandard_entities = [
    "Kosovo",
    "North Cyprus",
    "Somaliland",
]

In [30]:
hi_missing_iso_countries = (
    hi_df_clean.loc[hi_df_clean["iso_code"].isna(), "country"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

co2_countries = set(owid_co2_df_clean["country"].dropna().unique())

hi_missing_iso_in_co2 = sorted(set(hi_missing_iso_countries) & co2_countries)

display(pd.DataFrame({"hi_missing_iso_country_also_in_co2": hi_missing_iso_in_co2}))

,hi_missing_iso_country_also_in_co2


In [31]:
expected_iso_codes = ["COG", "COD", "CIV", "PRI"]

co2_iso_matches = (
    owid_co2_df_clean[
        owid_co2_df_clean["iso_code"].isin(expected_iso_codes)
    ][["country", "iso_code"]]
    .drop_duplicates()
    .sort_values("iso_code")
)

display(co2_iso_matches)

,country,iso_code
11246,Cote d'Ivoire,CIV
12395,Democratic Republic of Congo,COD
10621,Congo,COG


In [32]:
hi_iso_fixes = {
    "Ivory Coast": "CIV",         # CO2 name: Cote d'Ivoire
    "Congo Kinshasa": "COD",      # CO2 name: Democratic Republic of Congo
    "Congo Brazzaville": "COG",   # CO2 name: Congo
}

hi_df_clean["iso_code"] = hi_df_clean["iso_code"].fillna(
    hi_df_clean["country"].map(hi_iso_fixes)
)

display(
    hi_df_clean.loc[hi_df_clean["iso_code"].isna(), ["country", "year"]]
    .drop_duplicates()
    .sort_values(["country", "year"])
)

,country,year
760,Kosovo,2013
761,Kosovo,2015
762,Kosovo,2016
763,Kosovo,2017
764,Kosovo,2018
765,Kosovo,2019
766,Kosovo,2020
767,Kosovo,2021
1090,North Cyprus,2013
1091,North Cyprus,2015


So the remaining countries missing ISO codes are also missing from the CO2 dataset, these will be automatically dropped on left-merging through iso_code.

## 8. Select only desired columns from each dataset

In [33]:
print(owid_co2_df_clean.columns)
print(owid_energy_df_clean.columns)

Index(['country', 'year', 'iso_code', 'population', 'gdp', 'cement_co2',
       'cement_co2_per_capita', 'co2', 'co2_growth_abs', 'co2_growth_prct',
       'co2_including_luc', 'co2_including_luc_growth_abs',
       'co2_including_luc_growth_prct', 'co2_including_luc_per_capita',
       'co2_including_luc_per_gdp', 'co2_including_luc_per_unit_energy',
       'co2_per_capita', 'co2_per_gdp', 'co2_per_unit_energy', 'coal_co2',
       'coal_co2_per_capita', 'consumption_co2', 'consumption_co2_per_capita',
       'consumption_co2_per_gdp', 'cumulative_cement_co2', 'cumulative_co2',
       'cumulative_co2_including_luc', 'cumulative_coal_co2',
       'cumulative_flaring_co2', 'cumulative_gas_co2', 'cumulative_luc_co2',
       'cumulative_oil_co2', 'cumulative_other_co2', 'energy_per_capita',
       'energy_per_gdp', 'flaring_co2', 'flaring_co2_per_capita', 'gas_co2',
       'gas_co2_per_capita', 'ghg_excluding_lucf_per_capita', 'ghg_per_capita',
       'land_use_change_co2', 'land_use_chang

In [34]:
unique_iso = owid_co2_df_clean["iso_code"].unique()

# Filter both datasets to only include rows with ISO codes from the CO2 dataset
co2_filtered = owid_co2_df_clean[owid_co2_df_clean['iso_code'].isin(unique_iso)]
energy_filtered = owid_energy_df_clean[owid_energy_df_clean['iso_code'].isin(unique_iso)]

# Check for missing values and compare the population and gdp columns to pick one source to keep
print(f"OWID CO2 population and gdp columns have {co2_filtered['population'].isna().sum()} missing values and {co2_filtered['gdp'].isna().sum()} missing values")
print(f"OWID Energy population and gdp columns have {energy_filtered['population'].isna().sum()} missing values and {energy_filtered['gdp'].isna().sum()} missing values")

OWID CO2 population and gdp columns have 18 missing values and 486 missing values
OWID Energy population and gdp columns have 9 missing values and 378 missing values


Therefore we keep the population and gdp variables from the energy dataset to retain ma greater number of values.

In [35]:
owid_co2_df_clean = owid_co2_df_clean[['country', 'iso_code', 'year', 'co2_per_capita', 'consumption_co2_per_capita', 'energy_per_capita', 'temperature_change_from_co2', 'share_global_co2', 'land_use_change_co2_per_capita']]
owid_energy_df_clean = owid_energy_df_clean[['iso_code', 'year', 'population' , 'gdp', 'energy_per_capita', 'renewables_consumption']]

In [36]:
print(
    hi_df_clean.columns,
    mf_df_clean.columns,
    gini_df_clean.columns
)

Index(['country', 'year', 'happiness_index', 'happiness_index_rank',
       'iso_code'],
      dtype='str') Index(['iso_code', 'country', 'continent', 'hemisphere',
       'human_development_groups', 'undp_developing_regions', 'hdi_rank_2021',
       'year', 'material_footprint_per_capita'],
      dtype='str') Index(['country', 'iso_code', 'indicator_name', 'indicator_code', 'year',
       'gini_index'],
      dtype='str')


In [37]:
hi_df_clean = hi_df_clean[['iso_code', 'year', 'happiness_index' , 'happiness_index_rank']]
mf_df_clean = mf_df_clean[['iso_code', 'year', 'continent', 'hemisphere', 'human_development_groups', 'hdi_rank_2021', 'undp_developing_regions', 'material_footprint_per_capita']]
gini_df_clean = gini_df_clean[['iso_code', 'year', 'gini_index']]

## 9. Merge the datasets into a single raw *Global Wellbeing, Sustainability and Resource Use Dataset*

As illustrated by the different number of rows in each dataset, now that they cover the same year range this is due to different levels of country coverage. Since we are extracting the most variables from OWID CO2 data, we use this as the main template dataset on which to join the other datasets sequentially.

We directly drop countries not in the main dataset by using left-join.

In [38]:
from src.config import MERGED_RAW_PATH
from src.io import save_csv

df_merge1 = owid_co2_df_clean.merge(
    owid_energy_df_clean,
    on=["iso_code", "year"],
    how="left"
)

df_merge2 = df_merge1.merge(
    hi_df_clean,
    on=["iso_code", "year"],
    how="left"
)

df_merge3 = df_merge2.merge(
    mf_df_clean,
    on=["iso_code", "year"],
    how="left"
)

df_final_raw = df_merge3.merge(
    gini_df_clean,
    on=["iso_code", "year"],
    how="left"
)

# Sort the final data by country first and then by year
df_final_raw = df_final_raw.sort_values(by=["iso_code", "year"])

save_csv(df_final_raw,MERGED_RAW_PATH)

In [39]:
display(df_final_raw.head(10))
print(df_final_raw.columns)

,country,iso_code,year,co2_per_capita,consumption_co2_per_capita,energy_per_capita_x,temperature_change_from_co2,share_global_co2,land_use_change_co2_per_capita,population,...,renewables_consumption,happiness_index,happiness_index_rank,continent,hemisphere,human_development_groups,hdi_rank_2021,undp_developing_regions,material_footprint_per_capita,gini_index
90,Aruba,ABW,2013,8.395,NaN,47742.637,0.000,0.002,NaN,102570.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
91,Aruba,ABW,2014,8.435,NaN,47990.926,0.000,0.002,NaN,103381.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92,Aruba,ABW,2015,8.615,NaN,48905.531,0.000,0.003,NaN,104200.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
93,Aruba,ABW,2016,8.411,NaN,47619.418,0.000,0.002,NaN,104989.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
94,Aruba,ABW,2017,8.420,NaN,49061.195,0.000,0.002,NaN,105737.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95,Aruba,ABW,2018,8.158,NaN,46080.551,0.000,0.002,NaN,106447.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,Aruba,ABW,2019,7.767,NaN,43634.250,0.000,0.002,NaN,107089.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,Aruba,ABW,2020,7.641,NaN,42530.234,0.000,0.002,NaN,107411.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98,Aruba,ABW,2021,8.005,NaN,44880.773,0.000,0.002,NaN,107565.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,Afghanistan,AFG,2013,0.281,NaN,1007.797,0.001,0.025,0.002,31622709.0,...,NaN,4.04,143.0,Asia,Northern Hemisphere,Low,180.0,SA,1.88,NaN


Index(['country', 'iso_code', 'year', 'co2_per_capita',
       'consumption_co2_per_capita', 'energy_per_capita_x',
       'temperature_change_from_co2', 'share_global_co2',
       'land_use_change_co2_per_capita', 'population', 'gdp',
       'energy_per_capita_y', 'renewables_consumption', 'happiness_index',
       'happiness_index_rank', 'continent', 'hemisphere',
       'human_development_groups', 'hdi_rank_2021', 'undp_developing_regions',
       'material_footprint_per_capita', 'gini_index'],
      dtype='str')
